## SNP: remove multi-mapping variant (';' genes)

In [ ]:
import pandas as pd

snp = pd.read_csv("/data2/project/mjcho/project3_CYP/MODELING/Hirachial_Attention/Hiarachial_Processed_data/Cropped/SNP_Gene_Level_cropped.csv", index_col=0)
snp_filtered = snp.loc[:, ~snp.columns.str.contains(";", regex=False)]
snp_filtered.to_csv("/data3/project/yjok/CYP450/data/genomic_data/SNP+MET+RNA+CNV_refined/SNP_Gene_Level_cropped_refined.csv")

print(f"Saved SNP filtered data: {snp_filtered.shape[0]} samples x {snp_filtered.shape[1]} genes")

## MET: do not using 'contain'

In [5]:
import pandas as pd
import numpy as np

In [ ]:
# test.py -> contain 부분 수정

methyl = pd.read_csv("/data2/project/mjcho/project3_CYP/GDSC_genetic_data/METHYL/GSE68379_Matrix.processed.txt", sep = "\t")
methyl = methyl.T.copy()

methyl.reset_index(inplace=True)
GEO_list = methyl.loc[0]

methyl_beta = methyl[methyl['index'].str.endswith(".Beta")].copy()
methyl_beta.columns = GEO_list
methyl_beta["Row.names"] = methyl_beta["Row.names"].str.replace('_AVG.Beta', '', regex=False)

### filteration : Using Model name and change Index to model ID
approved_name = pd.read_csv("/data2/project/mjcho/project3_CYP/GDSC_data_preprocessing/preprocess_data/Approved_model_name_and_ID.csv")
model_list = pd.read_csv("/data2/project/mjcho/project3_CYP/GDSC_genetic_data/METHYL/model_list_20230608.csv")

# make id_name upper to use isin() methods
id_name = model_list[["model_id", "model_name"]]
id_name["model_name"] = id_name["model_name"].str.upper() 

# get the model ID using model name in methyl_beta dataframe
id_name_filtered = id_name[id_name["model_name"].isin(methyl_beta["Row.names"])] 
id_name_filtered = id_name_filtered[id_name_filtered["model_id"].isin(approved_name["Model ID"])]

idx = methyl_beta["Row.names"].str.upper().isin(id_name_filtered["model_name"])
methyl_beta_filtered = methyl_beta[idx]

methyl_beta_filtered.sort_values(by="Row.names", inplace=True)

id_name_filtered.sort_values(by="model_name", inplace=True)
# check if sorted values are same in two dataframes
print(sum(methyl_beta_filtered["Row.names"].values == id_name_filtered.model_name.values))
methyl_beta_filtered["model_id"] = id_name_filtered["model_id"]

id_name_filtered.model_id.reset_index(drop=True, inplace=True)
methyl_beta_filtered.reset_index(drop=True, inplace=True)
methyl_beta_filtered.model_id = id_name_filtered.model_id
methyl_beta_filtered.drop(columns='Row.names', inplace=True)
methyl_beta_filtered.set_index('model_id', inplace=True)



cpg_data = pd.read_csv("/data2/project/mjcho/project3_CYP/GDSC_genetic_data/METHYL/GPL13534-11288.txt", sep="\t", skiprows=37, low_memory=False)
ann = cpg_data[["ID", "UCSC_RefGene_Name"]].dropna().copy()

# using CpG site in methyl_beta_filtered
ann["ID"] = ann["ID"].astype(str)
methyl_beta_filtered.columns = methyl_beta_filtered.columns.astype(str)
ann = ann[ann["ID"].isin(methyl_beta_filtered.columns)].copy()

# UCSC_RefGene_Name split by ';' and explode
ann = ann.assign(Gene=ann["UCSC_RefGene_Name"].astype(str).str.split(";")).explode("Gene")
ann["Gene"] = ann["Gene"].astype(str).str.strip()
ann = ann[ann["Gene"].ne("")].copy()
ann = ann.drop_duplicates(subset=["ID", "Gene"])

# long format (sample, ID, beta)
methyl_beta_filtered.index.name = "model_id"
methyl_beta_filtered.columns.name = "ID"
long = methyl_beta_filtered.stack().rename("beta").reset_index()
# long columns: ["model_id", "ID", "beta"]
df = long.merge(ann[["ID", "Gene"]], on="ID", how="inner")  

# gene-level mean
target = (
    df.groupby(["model_id", "Gene"])["beta"]
      .mean()
      .unstack("Gene")          # rows=model_id, cols=Gene
)

# save
out_path = "/data3/project/yjok/CYP450/data/genomic_data/SNP+MET+RNA+CNV_refined/methylation_beta_gene_level.csv"
target.to_csv(out_path)

print("\nDone.")
print(f"Merged gene-level methylation: {target.shape[0]} models x {target.shape[1]} genes -> {out_path}")

In [6]:
# Data_Crop.ipynb -> Filtration Usable Samples

snp_idx = pd.read_csv("/data2/project/mjcho/project3_CYP/GDSC_preprocessed_data/All_genes_integrated/SNP_avsnp150_index.csv", index_col=0).values.flatten()
methyl_idx = pd.read_csv("/data2/project/mjcho/project3_CYP/GDSC_preprocessed_data/All_genes_integrated/methyl_beta_CpG_islands_index.csv", index_col=0).values.flatten()
cnv = pd.read_csv("/data2/project/mjcho/project3_CYP/GDSC_preprocessed_data/All_genes_integrated/CNV_all_genes.csv", low_memory=False, index_col=0)
cnv.sort_index(inplace=True)
cnv_idx = cnv.index.values
rna = pd.read_csv("/data2/project/mjcho/project3_CYP/GDSC_preprocessed_data/All_genes_integrated/RNA_tpm.csv", low_memory=False, index_col=0)
rna.sort_index(inplace=True)
rna_idx = rna.index.values


snp_set = set(snp_idx)
methyl_set = set(methyl_idx)
cnv_set = set(cnv_idx)
rna_set = set(rna_idx)


common_indices = rna_set.intersection(cnv_set, methyl_set, snp_set)
common_indices_array = np.array(list(common_indices))
print(len(common_indices))
common_indices_array.sort()
common_indices_array


methyl_gene_level = pd.read_csv("/data3/project/yjok/CYP450/data/genomic_data/SNP+MET+RNA+CNV_refined/methylation_beta_gene_level.csv")
methyl_gene_level.set_index('model_id', inplace=True)
methyl_gene_level.sort_index(axis=0, inplace=True)
methyl_gene_level.sort_index(axis=1, inplace=True)
methyl_refined = methyl_gene_level.loc[methyl_gene_level.index.isin(common_indices_array)]
methyl_refined.to_csv("/data3/project/yjok/CYP450/data/genomic_data/SNP+MET+RNA+CNV_refined/Methyl_Gene_Level_cropped_refined.csv", index=False)

print(f"Saved MET refined data: {methyl_refined.shape[0]} samples x {methyl_refined.shape[1]} genes")

877


array(['SIDM00003', 'SIDM00023', 'SIDM00042', 'SIDM00043', 'SIDM00044',
       'SIDM00045', 'SIDM00046', 'SIDM00047', 'SIDM00048', 'SIDM00052',
       'SIDM00053', 'SIDM00056', 'SIDM00064', 'SIDM00077', 'SIDM00078',
       'SIDM00079', 'SIDM00080', 'SIDM00081', 'SIDM00082', 'SIDM00083',
       'SIDM00084', 'SIDM00085', 'SIDM00086', 'SIDM00087', 'SIDM00088',
       'SIDM00090', 'SIDM00091', 'SIDM00092', 'SIDM00094', 'SIDM00095',
       'SIDM00096', 'SIDM00097', 'SIDM00105', 'SIDM00107', 'SIDM00108',
       'SIDM00111', 'SIDM00112', 'SIDM00113', 'SIDM00115', 'SIDM00116',
       'SIDM00117', 'SIDM00118', 'SIDM00119', 'SIDM00120', 'SIDM00121',
       'SIDM00122', 'SIDM00123', 'SIDM00124', 'SIDM00125', 'SIDM00127',
       'SIDM00128', 'SIDM00129', 'SIDM00130', 'SIDM00132', 'SIDM00134',
       'SIDM00135', 'SIDM00136', 'SIDM00137', 'SIDM00138', 'SIDM00139',
       'SIDM00140', 'SIDM00142', 'SIDM00143', 'SIDM00145', 'SIDM00146',
       'SIDM00148', 'SIDM00149', 'SIDM00150', 'SIDM00151', 'SIDM

## Multi-omics data processing

In [2]:
import pandas as pd
import numpy as np
import torch
from sklearn.preprocessing import StandardScaler

In [4]:
pgkb_available_gene = pd.read_csv("../gene_list.txt", index_col=0, header=None)
pgkb_gene_list = list(pgkb_available_gene.index)


df_snp = pd.read_csv("SNP_Gene_Level_count.csv")
df_met = pd.read_csv("MET_Gene_Level_mean.csv")
df_cnv = pd.read_csv("CNV.csv")
df_rna = pd.read_csv("RNA.csv")


snp_PGKB = df_snp.reindex(columns=pgkb_gene_list)
met_PGKB = df_met.reindex(columns=pgkb_gene_list)
cnv_PGKB = df_cnv.reindex(columns=pgkb_gene_list)
rna_PGKB = df_rna.reindex(columns=pgkb_gene_list)


# 69, 532, 740, 856 제거 (GDSC2에 없는 sample)
drop_indices = [69, 532, 740, 856]
snp_PGKB = snp_PGKB.drop(index=drop_indices)
met_PGKB = met_PGKB.drop(index=drop_indices)
cnv_PGKB = cnv_PGKB.drop(index=drop_indices)
rna_PGKB = rna_PGKB.drop(index=drop_indices)


snp_PGKB_imputed = snp_PGKB.fillna(0)
met_PGKB_imputed = met_PGKB.fillna(met_PGKB.mean())
cnv_PGKB_imputed = cnv_PGKB.fillna(2)
rna_PGKB_imputed = rna_PGKB.fillna(0)
rna_PGKB_log_imputed = rna_PGKB_imputed.apply(np.log1p)


snp_df_imputed = pd.DataFrame(snp_PGKB_imputed, columns=snp_PGKB.columns)
met_df_imputed = pd.DataFrame(met_PGKB_imputed, columns=met_PGKB.columns)
cnv_df_imputed = pd.DataFrame(cnv_PGKB_imputed, columns=cnv_PGKB.columns)
rna_df_imputed = pd.DataFrame(rna_PGKB_log_imputed, columns=rna_PGKB.columns)


snp_df_imputed.to_csv("SNP_PGKB.csv")
met_df_imputed.to_csv("MET_PGKB.csv")
cnv_df_imputed.to_csv("CNV_PGKB.csv")
rna_df_imputed.to_csv("RNA_PGKB.csv")

In [5]:
snp = pd.read_csv("SNP_PGKB.csv", index_col=0)
met = pd.read_csv("MET_PGKB.csv", index_col=0)
cnv = pd.read_csv("CNV_PGKB.csv", index_col=0)
rna = pd.read_csv("RNA_PGKB.csv", index_col=0)

result = []
meta_data = []
for gene in pgkb_gene_list : 
    snp_temp = snp[gene]
    met_temp = met[gene]
    cnv_temp = cnv[gene]
    rna_temp = rna[gene]

    gene_set = np.column_stack((snp_temp, met_temp, cnv_temp, rna_temp))
    feature_set = [f"{gene}_SNP", f"{gene}_MET", f"{gene}_CNV", f"{gene}_RNA"]

    meta_data.append({
        'Gene_ID' : gene,
        'Features' : feature_set,
        'No.Features' : len(feature_set)
    })
    result.append(torch.from_numpy(gene_set).type(torch.FloatTensor))
    print(gene,' : Gene set combined to result')

pd.DataFrame(meta_data).to_csv("PGKB_combined_by_gene_metadata.csv")
torch.save(result, "PGKB_combined_by_gene.pth")

AADAC  : Gene set combined to result
ABAT  : Gene set combined to result
ABCB1  : Gene set combined to result
ABCB11  : Gene set combined to result
ABCC1  : Gene set combined to result
ABCC10  : Gene set combined to result
ABCC2  : Gene set combined to result
ABCC3  : Gene set combined to result
ABCC4  : Gene set combined to result
ABCC5  : Gene set combined to result
ABCC8  : Gene set combined to result
ABCG1  : Gene set combined to result
ABCG2  : Gene set combined to result
ABCG5  : Gene set combined to result
ABCG8  : Gene set combined to result
ACACA  : Gene set combined to result
ACACB  : Gene set combined to result
ACADSB  : Gene set combined to result
ACBD3  : Gene set combined to result
ACE  : Gene set combined to result
ACE2  : Gene set combined to result
ACHE  : Gene set combined to result
ACO1  : Gene set combined to result
ACP1  : Gene set combined to result
ADA  : Gene set combined to result
ADAR  : Gene set combined to result
ADCY1  : Gene set combined to result
ADCY2  :

In [7]:
meta = pd.read_csv("PGKB_combined_by_gene_metadata.csv")
gene_data = torch.load("PGKB_combined_by_gene.pth")

gene_data_dict = {}
for i in range(len(gene_data)):
    dat = gene_data[i]
    gene_name = meta['Gene_ID'][i]
    feature_len = meta['No.Features'][i]
    gene_data_dict[gene_name] = dat

torch.save(gene_data_dict, "../PGKB_Gene_data_dict.pth")

255
254
